In [ ]:
#Reload notebook
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../../')

In [ ]:
# Comando para instalação da ultima versão estável do pyfortracc
!python -m pip install -qqq -U pyfortracc > /dev/null 2>&1 && echo "✅ pyfortracc instalado com sucesso!" || echo "❌ Erro na instalação"

In [ ]:
import pyfortracc

In [ ]:
# Data Source: Solar Dynamics Observatory (SDO) - AIA instrument
# Data URL: https://sdo.gsfc.nasa.gov/data/aiahmi/

In [ ]:
# Comando para instalação da ultima versão estável do pyfortracc
!python -m pip install -qqq -U gdown > /dev/null 2>&1 && echo "✅ gdown instalado com sucesso!" || echo "❌ Erro na instalação"

In [ ]:
import gdown, zipfile, os, shutil

# Remove the existing input files
shutil.rmtree('input', ignore_errors=True)

# Download the input files
url = 'https://drive.google.com/uc?id=1bBlmgjiZ9HyR_8mIkgoiVePMVkObwSYp'
gdown.download(url, 'input.zip', quiet=False)
with zipfile.ZipFile('input.zip', 'r') as zip_ref:
    for member in zip_ref.namelist():
        zip_ref.extract(member)
os.remove('input.zip')

In [ ]:
from PIL import Image
import numpy as np

def data(path, channel="B", radius_fraction=0.9):
    # Abre a imagem em RGB
    img = Image.open(path).convert("RGB")
    arr = np.array(img, dtype=float)  # float para suportar NaN

    # Seleciona o canal
    channel_map = {"R": 0, "G": 1, "B": 2}
    if channel not in channel_map:
        raise ValueError("Canal inválido. Use 'R', 'G' ou 'B'.")
    arr_channel = arr[:, :, channel_map[channel]]

    # Dimensões e centro
    h, w = arr_channel.shape
    cx, cy = w // 2, h // 2
    radius = int(min(cx, cy) * radius_fraction)

    # Máscara circular (disco solar)
    y, x = np.ogrid[:h, :w]
    mask = (x - cx)**2 + (y - cy)**2 <= radius**2

    # Aplica a máscara: pixels fora do círculo = NaN
    arr_circle = np.where(mask, arr_channel, np.nan)

    return arr_circle

In [ ]:
# Visualiza a animação dos frames de entrada
pyfortracc.plot_animation(path_files='input/*.jpg', read_function=data)

In [ ]:
# Define os parâmetros de entrada para o rastreamento
name_list = {}
name_list['input_path'] = 'input/' # Caminho para os arquivos de entrada
name_list['output_path'] = 'output/' # Caminho para os arquivos de saída
name_list['thresholds'] = [35] # Lista de limiares de intensidade a serem usados no processo de segmentação
name_list['min_cluster_size'] = [50] # Lista que contém o tamanho mínimo dos clusters
name_list['operator'] = '<=' # '>= - <=' ou '=='
name_list['min_overlap'] = 1 # Porcentagem mínima de sobreposição entre os clusters em dois frames consecutivos
name_list['timestamp_pattern'] = '%Y%m%d_%H0000_Ic_flat_1k.jpg' # Padrão de nome de arquivo de timestamp
name_list['delta_time'] = 60 * 6 # Intervalo de tempo entre os frames em minutos equivalente a 6 horas

In [ ]:
pyfortracc.track(name_list, data)

In [ ]:
import pandas as pd
import glob

tracking_files = sorted(glob.glob(name_list['output_path'] + '/track/trackingtable/*.parquet'))
tracking_table = pd.concat(pd.read_parquet(f) for f in tracking_files)
tracking_table.head()

In [ ]:
# Cria uma animação mostrando a evolução dos clusters ao longo do tempo
pyfortracc.plot_animation(figsize=(8, 8), # Tamanho da figura
                          name_list= name_list, # Passa a lista de nomes
                          read_function=data, # Passa a função de leitura
                          start_timestamp= tracking_table['timestamp'].min().strftime('%Y-%m-%d %H:%M:%S'), # Data e hora de início no formato 'YYYY-MM-DD HH:MM:SS'
                          end_timestamp= tracking_table['timestamp'].max().strftime('%Y-%m-%d %H:%M:%S'), # Data e hora de fim no formato 'YYYY-MM-DD HH:MM:SS'
                          info=True,
                          info_col_name=False,
                          info_cols=['uid'] # Colunas de informação a serem exibidas na animação
)